# BioPred Phase 1 -- Molecular Featurization

## Purpose

This notebook converts the validated molecule-level modeling table into structure-derived molecular feature artifacts for downstream machine learning.

Notebook 04 created the validated molecule-level modeling table. Notebook 05 defined the ML handoff policy that governs how the molecule table may be used downstream. Notebook 06 consumes both artifacts and generates model-input features from molecular structure only.

The central rule for this notebook is that the core model feature matrix must be derived from molecular structure. Labels, potency summaries, evidence diagnostics, record-conflict flags, molecule identifiers, and raw audit metadata must not be used as model features.

## Inputs

- Validated molecule-level modeling table from Notebook 04:
  - `chembl_gabaa_molecule_modeling_table.parquet`

- ML handoff policy JSON from Notebook 05:
  - defines the primary target, sensitivity labels, structure source, forbidden feature columns, diagnostic metadata, and identifier metadata

## Outputs

This notebook will generate and save:

- RDKit descriptor feature table
- Morgan fingerprint feature table
- Combined structure-derived feature matrix
- Primary target vector
- Sensitivity target table
- Modeling metadata table
- Featurization report JSON

## Objectives

1. Load the validated molecule-level modeling table from Notebook 4 and the ML handoff policy artifact from Notebook 5.
2. Resolve the saved Notebook 5 ML handoff policy into local variables needed for molecular featurization. Perform a lightweight column-presence check, then proceed to structure parsing.
3. Partition the molecule table into modeling roles: structure source, target columns, sensitivity label columns, metadata columns, and forbidden feature columns.
4. Parse the policy-defined structure source column into RDKit molecule objects, while preserving row alignment between `X`, `y`, sensitivity labels, and metadata.
5. Generate deterministic RDKit molecular descriptor features from parsed molecule objects.
6. Generate fixed-length Morgan fingerprint features from parsed molecule objects using documented fingerprint settings.
7. Assemble aligned structure-derived model-input artifacts: feature matrix `X`, primary target `y`, sensitivity labels, and modeling metadata.
8. Save the feature, target, metadata, and featurization report artifacts under the processed data directory.


In [1]:
# list imports needed for this notebook
from pathlib import Path
import sys
import os
import json
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs

# Force notebook runtime to project root
%cd /home/ryanm/projects/BioPred

# define paths and link src.paths
SRC_DIR = Path.cwd() / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import biopred.paths as paths

/home/ryanm/projects/BioPred


In [2]:
# import our subject files and verify the contents before continuing.
MOLECULE_TABLE_PATH = paths.PROCESSED_DATA_DIR / "chembl_gabaa_molecule_modeling_table.parquet"
HANDOFF_POLICY_PATH = paths.PROCESSED_DATA_DIR / "ml_handoff_policy.json"

# verify paths
print(MOLECULE_TABLE_PATH, HANDOFF_POLICY_PATH)

# load inputs
molecule_df = pd.read_parquet(MOLECULE_TABLE_PATH)

with open(HANDOFF_POLICY_PATH, "r") as f:
    handoff_policy = json.load(f)

print(f"Molecule table shape: {molecule_df.shape}")
print(f"Handoff policy type {type(handoff_policy)}")


/home/ryanm/projects/BioPred/data/processed/chembl_gabaa_molecule_modeling_table.parquet /home/ryanm/projects/BioPred/data/processed/ml_handoff_policy.json
Molecule table shape: (1546, 21)
Handoff policy type <class 'dict'>


In [3]:
# consume Notebook 5 handoff policy, keeping it compact as its just a simple validation check.

PRIMARY_TARGET = handoff_policy["primary_modeling_label"]

SENSITIVITY_LABELS = handoff_policy["sensitivity_label"]

TARGET_COLUMNS = handoff_policy["target_columns"]

CONFLICT_FLAG_COLUMNS = handoff_policy["conflict_flag_columns"]

STRUCTURE_SOURCE = handoff_policy["structure_source_columns"]

FORBIDDEN_FEATURE_COLUMNS = handoff_policy["forbidden_feature_columns"]

DIAGNOSTIC_METADATA_COLUMNS = handoff_policy["diagnostic_metadata_columns"]

IDENTIFIER_METADATA_COLUMNS = handoff_policy["identifier_metadata_cols"]

required_policy_columns = set(
    [STRUCTURE_SOURCE]
    + TARGET_COLUMNS
    + CONFLICT_FLAG_COLUMNS
    + FORBIDDEN_FEATURE_COLUMNS
    + DIAGNOSTIC_METADATA_COLUMNS
    + IDENTIFIER_METADATA_COLUMNS
)

missing_policy_columns = required_policy_columns - set(molecule_df.columns)

if missing_policy_columns:
    raise ValueError(
        f"Missing policy-defined columns: {sorted(missing_policy_columns)}"
    )

print("Notebook 5 policy consumed successfully.")
print(f"Primary target: {PRIMARY_TARGET}")
print(f"Structure source: {STRUCTURE_SOURCE}")
print(f"Required policy columns present: {len(required_policy_columns)}") 



Notebook 5 policy consumed successfully.
Primary target: active_median_px_6_0
Structure source: canonical_smiles
Required policy columns present: 21


### **Section 2-3 Summary**

The Notebook 5 ML handoff policy was consumed and mapped into local Notebook 6 variables. The required policy-defined columns were confirmed to be present in the molecule-level modeling table.

The molecule table was then organized into downstream roles:

- primary modeling label: `active_median_px_6_0`
- sensitivity-analysis labels: `active_median_px_5_5` and `active_median_px_6_5`
- structure source column: `canonical_smiles`
- metadata columns: molecule identifiers, evidence diagnostics, and conflict flags
- forbidden feature columns: potency summaries and activity labels that must not be used as model inputs

At this point, no molecular features have been generated yet. The important outcome is that Notebook 6 has consumed the Notebook 5 contract and identified which parts of the molecule table will be used for targets, traceability, diagnostics, and molecular feature generation.

The next section begins the first true featurization step by converting `canonical_smiles` into RDKit molecule objects.

### **Section 4: Parse molecular structures**

This section converts the policy-defined structure source column, `canonical_smiles`, into RDKit molecule objects.

The raw SMILES string is retained as source metadata, but core molecular features will be generated from parsed RDKit molecules. Row alignment must be preserved so that each molecule object remains matched to its corresponding labels and metadata.

Invalid or unparseable SMILES will cause the notebook to stop rather than silently dropping rows.

In [4]:
# convert policy-defined SMILES column into RDKit Mol objects
smiles_series = molecule_df[STRUCTURE_SOURCE]

mol_series = smiles_series.apply(Chem.MolFromSmiles)

# check new series for missingness and invalid molecules
invalid_mol_mask = mol_series.isna()
n_invalid_mols = invalid_mol_mask.sum()

print(f"Parsed molecules: {mol_series.notna().sum():,}")
print(f"Invalid molecules: {n_invalid_mols:,}")

mol_series.head()

Parsed molecules: 1,546
Invalid molecules: 0


0    <rdkit.Chem.rdchem.Mol object at 0x727c8a738f20>
1    <rdkit.Chem.rdchem.Mol object at 0x727c8a7390e0>
2    <rdkit.Chem.rdchem.Mol object at 0x727c8a739620>
3    <rdkit.Chem.rdchem.Mol object at 0x727c8a739700>
4    <rdkit.Chem.rdchem.Mol object at 0x727c8a739770>
Name: canonical_smiles, dtype: object

### **Section 4 Summary**

The policy-defined structure source column, `canonical_smiles`, was successfully parsed into RDKit molecule objects.

All 1,546 molecule-level rows produced valid RDKit molecules, and no invalid or unparseable SMILES were detected. The parsed molecule series retains the original molecule table index, preserving alignment with labels and metadata for downstream feature generation.

### **Section 5: Generate RDKit descriptor features**

This section generates deterministic RDKit molecular descriptor features from the parsed molecule objects.

RDKit descriptors provide numeric molecular properties derived from structure, such as molecular weight, topological properties, charge-related descriptors, surface-area descriptors, and graph-based indices. These features are allowed under the Notebook 5 handoff policy because they are computed from molecular structure rather than potency summaries, labels, diagnostics, or identifiers.

The output of this section will be a row-aligned descriptor table with one row per molecule and one column per RDKit descriptor.

In [5]:
# first look at the Chem library's Descriptor registry
descriptor_registry = Descriptors._descList

descriptor_names = [name for name in descriptor_registry]

print(f"Number of RDKit descriptors available: {len(descriptor_names)}")
descriptor_names[:15]

Number of RDKit descriptors available: 217


[('MaxAbsEStateIndex',
  <function rdkit.Chem.EState.EState.MaxAbsEStateIndex(mol, force=1)>),
 ('MaxEStateIndex',
  <function rdkit.Chem.EState.EState.MaxEStateIndex(mol, force=1)>),
 ('MinAbsEStateIndex',
  <function rdkit.Chem.EState.EState.MinAbsEStateIndex(mol, force=1)>),
 ('MinEStateIndex',
  <function rdkit.Chem.EState.EState.MinEStateIndex(mol, force=1)>),
 ('qed',
  <function rdkit.Chem.QED.qed(mol, w=QEDproperties(MW=0.66, ALOGP=0.46, HBA=0.05, HBD=0.61, PSA=0.06, ROTB=0.65, AROM=0.48, ALERTS=0.95), qedProperties=None)>),
 ('SPS', <function rdkit.Chem.SpacialScore.SPS(mol, normalize=True)>),
 ('MolWt', <function rdkit.Chem.Descriptors.<lambda>(*x, **y)>),
 ('HeavyAtomMolWt', <function rdkit.Chem.Descriptors.HeavyAtomMolWt(x)>),
 ('ExactMolWt', <function rdkit.Chem.Descriptors.<lambda>(*x, **y)>),
 ('NumValenceElectrons',
  <function rdkit.Chem.Descriptors.NumValenceElectrons(mol)>),
 ('NumRadicalElectrons',
  <function rdkit.Chem.Descriptors.NumRadicalElectrons(mol)>),
 ('Ma

In [6]:
# compute descriptors for a single molecule and test before we add for all in our molecule_df

def compute_rdkit_descriptors(mol, descriptor_registry):
    """
    Compute RDKit descriptors for one molecule.

    Returns
    ----------
    dict
        decriptor_name -> descriptor_value
    """
    descriptor_values = {}

    for name, fn in descriptor_registry:
        try:
            descriptor_values[name] = fn(mol)
        except Exception:
            descriptor_values[name] = np.nan
    
    return descriptor_values

# test the function
example_mol = mol_series.iloc[0]

example_descriptors = compute_rdkit_descriptors(
    mol = example_mol,
    descriptor_registry = descriptor_registry
)

print(f"Descriptors computed for one molecule: {len(example_descriptors)}")
print(list(example_descriptors.items())[:10])

# then one more quick check before the full table
example_missing = pd.Series(example_descriptors).isna().sum()
print(f"Missing descriptor values for example molecule: {example_missing}")

Descriptors computed for one molecule: 217
[('MaxAbsEStateIndex', np.float64(12.479469246031746)), ('MaxEStateIndex', np.float64(12.479469246031746)), ('MinAbsEStateIndex', np.float64(0.15647326677584594)), ('MinEStateIndex', np.float64(-0.4911921208812735)), ('qed', 0.7966555038307888), ('SPS', 13.409090909090908), ('MolWt', 319.748), ('HeavyAtomMolWt', 305.636), ('ExactMolWt', 319.072368988), ('NumValenceElectrons', 114)]
Missing descriptor values for example molecule: 0


In [7]:
# now compute for all molecules
rdkit_decriptor_records = [
    compute_rdkit_descriptors(mol, descriptor_registry)
    for mol in mol_series
]

rdkit_descriptor_df = pd.DataFrame(
    rdkit_decriptor_records,
    index = mol_series.index
)

# make an index prefix so we know what their origin is downstream
rdkit_descriptor_df = rdkit_descriptor_df.add_prefix("rdkit_")

print(f"RDKit descriptor table shape: {rdkit_descriptor_df.shape}") # Expecting (1546, 217)
rdkit_descriptor_df.head()



RDKit descriptor table shape: (1546, 217)


,rdkit_MaxAbsEStateIndex,rdkit_MaxEStateIndex,rdkit_MinAbsEStateIndex,rdkit_MinEStateIndex,rdkit_qed,rdkit_SPS,rdkit_MolWt,rdkit_HeavyAtomMolWt,rdkit_ExactMolWt,rdkit_NumValenceElectrons,...,rdkit_fr_sulfide,rdkit_fr_sulfonamd,rdkit_fr_sulfone,rdkit_fr_term_acetylene,rdkit_fr_tetrazole,rdkit_fr_thiazole,rdkit_fr_thiocyan,rdkit_fr_thiophene,rdkit_fr_unbrch_alkane,rdkit_fr_urea
0,12.479469,12.479469,0.156473,-0.491192,0.796656,13.409091,319.748,305.636,319.072369,114,...,0,0,0,0,0,0,0,0,0,0
1,13.467739,13.467739,0.165440,-0.539203,0.793248,13.409091,303.293,289.181,303.101920,114,...,0,0,0,0,0,0,0,0,0,0
2,12.024468,12.024468,0.028237,-0.028237,0.791645,14.600000,284.746,271.642,284.071641,100,...,0,0,0,0,0,0,0,0,0,0
3,12.607591,12.607591,0.183771,-0.526621,0.373030,12.750000,326.316,312.204,326.112738,122,...,0,0,0,0,0,0,0,0,0,0
4,12.710678,12.710678,0.172729,-0.650654,0.355901,13.384615,354.370,336.226,354.144038,134,...,0,0,0,0,0,0,0,0,0,0


In [8]:
# check and validate the information contained in our new df
n_missing_descriptor_values = rdkit_descriptor_df.isna().sum().sum()
n_decriptor_columns_with_missing = (rdkit_descriptor_df.isna().sum() > 0).sum()

print(f"Missing descriptor values: {n_missing_descriptor_values:,}")
print(f"Descriptor columns with missing values: {n_missing_descriptor_values:,}")

Missing descriptor values: 32
Descriptor columns with missing values: 32


The missingness check located 32 missing values across 32 different columns in our newly-created rdkit_descriptor_df.  It is likely that one molecule failed 32 descriptor calculations, though we don't know that yet.  Rather than just drop or impute even though its a small amount of data we will investigate further.

In [9]:
# missingness by molecule row
missing_by_row = rdkit_descriptor_df.isna().sum(axis = 1)

# missingness by descriptor column
missing_by_col = rdkit_descriptor_df.isna().sum(axis = 0)

# print statements and defined checks
print(f"Rows with missing descriptor values: {(missing_by_row > 0).sum()}")
print(f"Cols with missing descriptor values: {(missing_by_col > 0).sum()}")

missing_rows = missing_by_row[missing_by_row > 0]
missing_cols = missing_by_col[missing_by_col > 0]

missing_rows, missing_cols.sort_values(ascending = False).head(40) 


Rows with missing descriptor values: 3
Cols with missing descriptor values: 12


(117    12
 124    12
 660     8
 dtype: int64,
 rdkit_BCUT2D_CHGLO           3
 rdkit_BCUT2D_CHGHI           3
 rdkit_BCUT2D_MWLOW           3
 rdkit_BCUT2D_MWHI            3
 rdkit_BCUT2D_LOGPHI          3
 rdkit_BCUT2D_LOGPLOW         3
 rdkit_BCUT2D_MRHI            3
 rdkit_BCUT2D_MRLOW           3
 rdkit_MinAbsPartialCharge    2
 rdkit_MaxAbsPartialCharge    2
 rdkit_MinPartialCharge       2
 rdkit_MaxPartialCharge       2
 dtype: int64)

In [10]:
# one more inspection cell, we want to identify the affected molecules
problem_row_indices = missing_rows.index.tolist()

molecule_df.loc[
    problem_row_indices,
    [STRUCTURE_SOURCE, *IDENTIFIER_METADATA_COLUMNS]
]



,canonical_smiles,molregno,molecule_chembl_id,standard_inchi_key
117,NCCC[Se](=O)O,59510,CHEMBL41065,WTWJERJTGDRNRR-UHFFFAOYSA-N
124,O=[Se](O)C1CCNCC1,60410,CHEMBL290961,IFVWTFDUNUGVFM-UHFFFAOYSA-N
660,O=C([O-])CCC/N=C(\c1ccc(Cl)cc1)c1cc(F)ccc1O.[Na+],268340,CHEMBL160058,IIZZSPOBOIMENB-SJEOTZHBSA-M


In [11]:
# and then store the failed descriptor names for later
failed_descriptor_cols = missing_cols.index.tolist()

failed_descriptor_cols

['rdkit_MaxPartialCharge',
 'rdkit_MinPartialCharge',
 'rdkit_MaxAbsPartialCharge',
 'rdkit_MinAbsPartialCharge',
 'rdkit_BCUT2D_MWHI',
 'rdkit_BCUT2D_MWLOW',
 'rdkit_BCUT2D_CHGHI',
 'rdkit_BCUT2D_CHGLO',
 'rdkit_BCUT2D_LOGPHI',
 'rdkit_BCUT2D_LOGPLOW',
 'rdkit_BCUT2D_MRHI',
 'rdkit_BCUT2D_MRLOW']

### **Section 5 Summary**

RDKit molecular descriptors were generated for all parsed molecule objects.

The descriptor registry contained 217 RDKit descriptors, producing a row-aligned descriptor table with 1,546 molecules and 217 descriptor columns. Descriptor calculation succeeded broadly, with 32 missing descriptor values across 3 molecule rows and 12 descriptor columns.

Missing descriptor values were limited to BCUT2D and partial-charge descriptor families. No rows or descriptor columns were dropped in this notebook. Missing-value handling is deferred to Notebook 7, where preprocessing decisions can be fit on training data only.

### **Section 6: Generate Morgan fingerprint features**

This section generates fixed-length Morgan fingerprint features from the parsed RDKit molecule objects.

Morgan fingerprints represent molecular structure through hashed circular atom-neighborhood patterns. For BioPred v1, this notebook uses a 2-radius, 2,048-bit binary fingerprint configuration as a conventional small-molecule ML baseline.

The output of this section will be a row-aligned fingerprint table with one row per molecule and one binary column per fingerprint bit. Basic fingerprint QA will confirm the expected table shape, binary values, bit density, and unused bit count.

No target labels, potency summaries, identifiers, diagnostics, or conflict flags are used in fingerprint generation.

In [12]:
# setting up the config for our fingerprints
MORGAN_RADIUS = 2
MORGAN_N_BITS = 2048
MORGAN_USE_COUNTS = False

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius = MORGAN_RADIUS,
    fpSize = MORGAN_N_BITS,
    countSimulation = MORGAN_USE_COUNTS,
)

print(f"Morgan radius: {MORGAN_RADIUS}")
print(f"Morgan bits: {MORGAN_N_BITS}")
print(f"Morgan count simulation: {MORGAN_USE_COUNTS}")

Morgan radius: 2
Morgan bits: 2048
Morgan count simulation: False


In [13]:
# function we will use for our fingerprint generation
def mol_to_morgan_array(mol, generator, n_bits):
    """
    Convert one RDKit Mol into a fixed-length Morgan fingerprint array.

    Returns
    ----------
    np.ndarray
        Binary fingerprint array with shape (n_bits,).
    """

    arr = np.zeros((n_bits,), dtype = np.int8)

# generate fingerprint object from mol.
    fp = generator.GetFingerprint(mol)

# convert the RDKit fingerprint object into numpy array.
    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr

In [14]:
# test on one mol
example_fp = mol_to_morgan_array(
    mol = mol_series.iloc[0],
    generator = morgan_generator,
    n_bits = MORGAN_N_BITS
)

print("Fingerprint shape:", example_fp.shape)
print("Unique values:", np.unique(example_fp))
print("Bits on:", example_fp.sum())

Fingerprint shape: (2048,)
Unique values: [0 1]
Bits on: 49


In [15]:
# good output, now we can build for all molecules in molecule_df
morgan_arrays = np.vstack([
    mol_to_morgan_array(
        mol = mol,
        generator = morgan_generator,
        n_bits = MORGAN_N_BITS,
    )
    for mol in mol_series
])

morgan_fp_df = pd.DataFrame(
    morgan_arrays,
    index = mol_series.index,
    columns = [f"morgan_{i}" for i in range(MORGAN_N_BITS)],
)

print(f"Morgan fingerprint table shape: {morgan_fp_df.shape}")
morgan_fp_df.head()


Morgan fingerprint table shape: (1546, 2048)


,morgan_0,morgan_1,morgan_2,morgan_3,morgan_4,morgan_5,morgan_6,morgan_7,morgan_8,morgan_9,...,morgan_2038,morgan_2039,morgan_2040,morgan_2041,morgan_2042,morgan_2043,morgan_2044,morgan_2045,morgan_2046,morgan_2047
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
# Running QA checks on the new morgan_fp_df to make sure its ok
unique_fp_values = np.unique(morgan_fp_df.to_numpy())

mean_bits_on = morgan_fp_df.sum(axis = 1).mean()
min_bits_on = morgan_fp_df.sum(axis = 1).min()
max_bits_on = morgan_fp_df.sum(axis = 1).max()

dead_bits = (morgan_fp_df.sum(axis = 0) == 0).sum()

print(f"Unique fingerprint values: {unique_fp_values}")
print(f"Mean bits on per molecule: {mean_bits_on:.2f}")
print(f"Min bits on per molecule: {min_bits_on}")
print(f"Max bits on per molecule: {max_bits_on}")
print(f"Dead fingerprint bits: {dead_bits}")

Unique fingerprint values: [0 1]
Mean bits on per molecule: 44.69
Min bits on per molecule: 5
Max bits on per molecule: 86
Dead fingerprint bits: 151


### **Section 6 Summary**

Morgan fingerprint features were generated for all parsed molecule objects using a radius-2, 2,048-bit binary fingerprint configuration.

The resulting fingerprint table contains 1,546 molecule rows and 2,048 fingerprint bit columns. Fingerprint QA confirmed binary values only, with an average of 44.69 active bits per molecule. The sparsest molecule activated 5 bits and the densest activated 86 bits.

A total of 151 fingerprint bits were inactive across the full dataset. These columns were retained in the raw fingerprint artifact. Any variance filtering or dead-bit removal is deferred to Notebook 7 so that preprocessing decisions can be fit on training data only.

### **Section 7: Assemble aligned model-input artifacts**

This section assembles the structure-derived feature blocks into a single model-input feature matrix and prepares the target and metadata artifacts for downstream notebooks.

The feature matrix is built only from RDKit descriptors and Morgan fingerprints generated in Notebook 6. The primary target, sensitivity labels, molecule identifiers, structure source, evidence diagnostics, and conflict flags are retained as separate artifacts and are not included in the feature matrix.

The key check in this section is alignment: the feature matrix, primary target, sensitivity labels, and metadata must all preserve the same row index and molecule ordering.

In [17]:
# combine our feature blocks we created here in this notebook, in rdkit_descriptor_df and morgan_fp_df.
X = pd.concat([rdkit_descriptor_df, morgan_fp_df], axis = 1)

# build target and metadata artifacts
y_primary = molecule_df[[PRIMARY_TARGET]].copy()
y_sensitivity = molecule_df[SENSITIVITY_LABELS].copy()

# metadata_cols here will be our list from earlier in this ntbk - (forbidden_cols + target_cols)
metadata_columns = sorted(
    set(
        [STRUCTURE_SOURCE]
        + CONFLICT_FLAG_COLUMNS
        + DIAGNOSTIC_METADATA_COLUMNS
        + IDENTIFIER_METADATA_COLUMNS
    )
)

metadata = molecule_df[metadata_columns].copy()

# verify
print("X shape:", X.shape)
print("Primary target shape:", y_primary.shape)
print("Sensitivity labels shape:", y_sensitivity.shape)
print("Metadata shape:", metadata.shape)


X shape: (1546, 2265)
Primary target shape: (1546, 1)
Sensitivity labels shape: (1546, 2)
Metadata shape: (1546, 12)


In [18]:
# confirm row alignment across assembled artifacts
assert X.index.equals(molecule_df.index)
assert y_primary.index.equals(molecule_df.index)
assert y_sensitivity.index.equals(molecule_df.index)
assert metadata.index.equals(molecule_df.index)

# confirm policy-forbidden raw columns did not enter X
forbidden_overlap = set(X.columns) & set(FORBIDDEN_FEATURE_COLUMNS)

if forbidden_overlap:
    raise ValueError(
        f"Forbidden policy columns found in X: {sorted(forbidden_overlap)}"
    )

print("Artifacts are row-aligned.")
print("No forbidden policy columns found in X.")

Artifacts are row-aligned.
No forbidden policy columns found in X.


### **Section 7 Summary**

The RDKit descriptor and Morgan fingerprint feature blocks were assembled into a single structure-derived feature matrix.

The final feature matrix contains 1,546 molecules and 2,265 features: 217 RDKit descriptor columns and 2,048 Morgan fingerprint columns. The primary target, sensitivity labels, and modeling metadata were retained as separate aligned artifacts.

Row alignment was confirmed across the feature matrix, primary target, sensitivity labels, and metadata. No policy-forbidden raw columns were found in the feature matrix.

### **Section 8: Save molecular feature artifacts**

This section saves the aligned molecular feature, target, sensitivity-label, metadata, and featurization-report artifacts for downstream notebooks.

Notebook 6 outputs are treated as processed data assets, not fitted model artifacts. They will be saved under `data/processed/features/` so they remain separate from upstream molecule-level source artifacts and later modeling artifacts such as fitted preprocessors, trained models, metrics, and reports.

Notebook 7 will load these saved artifacts for feature QA, split-aware preprocessing, and train/test split construction.

In [24]:
# define our dir path for all of the feature output we will be sending.  
FEATURE_DIR = paths.PROCESSED_DATA_DIR / "features"

# define the output paths for the files to be sent 
RDKIT_DESCRIPTOR_PATH = FEATURE_DIR / "chembl_gabaa_rdkit_descriptors.parquet"
MORGAN_FP_PATH = FEATURE_DIR / "chembl_gabaa_morgan_fingerprints.parquet"
FEATURE_MATRIX_PATH = FEATURE_DIR / "chembl_gabaa_feature_matrix.parquet"

PRIMARY_TARGET_PATH = FEATURE_DIR / "chembl_gabaa_primary_target.parquet"
SENSITIVITY_LABELS_PATH = FEATURE_DIR / "chembl_gabaa_sensitivity_labels.parquet"
METADATA_PATH = FEATURE_DIR / "chembl_gabaa_modeling_metadata.parquet"

#just creating the featurization report path, we will populate it at the end of section 8
FEATURIZATION_REPORT_PATH = FEATURE_DIR / "chembl_gabaa_featurization_report.json"

artifact_paths = {
    "rdkit_descriptors" : RDKIT_DESCRIPTOR_PATH,
    "morgan_fingerprints" : MORGAN_FP_PATH,
    "feature_matrix" : FEATURE_MATRIX_PATH,
    "primary_target" : PRIMARY_TARGET_PATH,
    "sensitivity_labels" : SENSITIVITY_LABELS_PATH,
    "metadata" : METADATA_PATH,
    "featurization_report" : FEATURIZATION_REPORT_PATH,
}

# check and verify pathing
artifact_paths

{'rdkit_descriptors': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_rdkit_descriptors.parquet'),
 'morgan_fingerprints': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_morgan_fingerprints.parquet'),
 'feature_matrix': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_feature_matrix.parquet'),
 'primary_target': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_primary_target.parquet'),
 'sensitivity_labels': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_sensitivity_labels.parquet'),
 'metadata': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_modeling_metadata.parquet'),
 'featurization_report': PosixPath('/home/ryanm/projects/BioPred/data/processed/features/chembl_gabaa_featurization_report.json')}

In [25]:
# now save the processed feature artifacts, preserving indices.
rdkit_descriptor_df.to_parquet(RDKIT_DESCRIPTOR_PATH)
morgan_fp_df.to_parquet(MORGAN_FP_PATH)
X.to_parquet(FEATURE_MATRIX_PATH)
y_primary.to_parquet(PRIMARY_TARGET_PATH)
y_sensitivity.to_parquet(SENSITIVITY_LABELS_PATH)
metadata.to_parquet(METADATA_PATH)


In [21]:
# populate our featurization_report now, giving a summary of everything that will be in the features/ folder 
featurization_report = {
    "input_molecule_table_path" : str(MOLECULE_TABLE_PATH),
    "handoff_policy_path" : str(HANDOFF_POLICY_PATH),
    "feature_dir" : str(FEATURE_DIR),

    "structure_source" : STRUCTURE_SOURCE,
    "primary_target" : PRIMARY_TARGET,
    "sensitivity_labels" : SENSITIVITY_LABELS,

    "n_molecules" : int(molecule_df.shape[0]),
    "n_rdkit_descriptors" : int(rdkit_descriptor_df.shape[1]),
    "n_morgan_bits" : int(MORGAN_N_BITS),
    "n_total_features" : int(X.shape[1]),

    "morgan_radius" : int(MORGAN_RADIUS),
    "morgan_use_counts" : bool(MORGAN_USE_COUNTS),

    "n_invalid_molecules" : int(n_invalid_mols),
    "n_missing_rdkit_descriptor_values" : int(rdkit_descriptor_df.isna().sum().sum()),
    "n_rdkit_descriptor_columns_with_missing" : int((rdkit_descriptor_df.isna().sum() > 0).sum()),
    "n_dead_morgan_bits" : int((morgan_fp_df.sum(axis = 0) == 0).sum()),

    "artifact_paths" : {
        key : str(path)
        for key, path in artifact_paths.items()
    },

}

In [22]:
# now save it to populate
with open(FEATURIZATION_REPORT_PATH, "w") as f:
    json.dump(featurization_report, f, indent = 2)

In [26]:
# Reload saved artifacts for a lightweight smoke check.

X_reload = pd.read_parquet(FEATURE_MATRIX_PATH)
y_primary_reload = pd.read_parquet(PRIMARY_TARGET_PATH)
y_sensitivity_reload = pd.read_parquet(SENSITIVITY_LABELS_PATH)
metadata_reload = pd.read_parquet(METADATA_PATH)

with open(FEATURIZATION_REPORT_PATH, "r") as f:
    featurization_report_reload = json.load(f)

print("X reload shape:", X_reload.shape)
print("Primary target reload shape:", y_primary_reload.shape)
print("Sensitivity labels reload shape:", y_sensitivity_reload.shape)
print("Metadata reload shape:", metadata_reload.shape)

X reload shape: (1546, 2265)
Primary target reload shape: (1546, 1)
Sensitivity labels reload shape: (1546, 2)
Metadata reload shape: (1546, 12)


### **Notebook 6 Summary**

Notebook 6 generated molecular feature artifacts from the validated molecule-level modeling table and the Notebook 5 ML handoff policy.

The molecule table and handoff policy were loaded successfully, and the policy-defined structure source, target labels, metadata columns, and forbidden feature columns were consumed for downstream featurization. The structure source column, `canonical_smiles`, was parsed into RDKit molecule objects with no invalid molecules detected.

Two structure-derived feature families were generated:

- RDKit molecular descriptors: 217 descriptor columns
- Morgan fingerprints: 2,048 binary fingerprint columns using radius 2

The final combined feature matrix contains 1,546 molecules and 2,265 structure-derived features. The primary target, sensitivity labels, and modeling metadata were saved as separate row-aligned artifacts.

Feature QA identified 32 missing RDKit descriptor values across 3 molecule rows and 12 descriptor columns, limited to BCUT2D and partial-charge descriptor families. Morgan fingerprint QA confirmed binary values only, with an average of 44.69 active bits per molecule and 151 inactive fingerprint bits across the dataset. No rows or feature columns were removed in this notebook.

Notebook 6 saved the processed feature artifacts under `data/processed/features/` and wrote a compact featurization report JSON summarizing the feature-generation configuration, artifact paths, and QA counts. A final reload smoke check confirmed that the saved artifacts are readable and preserve expected shapes and row alignment.

Notebook 7 should load these artifacts for feature QA, split-aware preprocessing, and train/test split construction. Imputation, variance filtering, scaling, feature pruning, and other preprocessing decisions should be fit on training data only.